### Stage 1 — Incremental batch: land a bronze table

Runs weekly. Each run fetches the forecast for **today + the next 6 days** (a 7-day window) and
**appends** it to a bronze table in Unity Catalog with a `fetched_at` timestamp.

Because windows shift every week, consecutive runs overlap on shared days — the same hour can
appear in multiple snapshots with revised values. Bronze only aims to *minimize* duplicates,
not eliminate them; full dedup and reconciliation (latest value per hour) is the silver layer's job.

In [0]:
from open_meteo.client import Location, fetch_hourly_forecast

CITY_DIM_TABLE = "workspace.open_meteo.dim_city"
BRONZE_TABLE = "workspace.open_meteo.bronze_hourly_forecast"

BRONZE_COLUMNS = ", ".join(
    (
        "latitude DOUBLE",
        "longitude DOUBLE",
        "forecast_time TIMESTAMP",
        "temperature_2m DOUBLE",
        "temperature_2m_unit STRING",
        "apparent_temperature DOUBLE",
        "apparent_temperature_unit STRING",
        "precipitation DOUBLE",
        "precipitation_unit STRING",
        "wind_speed_10m DOUBLE",
        "wind_speed_10m_unit STRING",
        "weather_code INT",
        "fetched_at TIMESTAMP",
    )
)

STAGING_COLUMNS = ", ".join(
    (
        "latitude DOUBLE",
        "longitude DOUBLE",
        "timezone STRING",
        "hourly_units STRUCT<time: STRING, temperature_2m: STRING,"
        " precipitation: STRING, weather_code: STRING,"
        " wind_speed_10m: STRING, apparent_temperature: STRING>",
        "hourly STRUCT<time: ARRAY<TIMESTAMP>, temperature_2m: ARRAY<DOUBLE>,"
        " precipitation: ARRAY<DOUBLE>, weather_code: ARRAY<INT>,"
        " wind_speed_10m: ARRAY<DOUBLE>, apparent_temperature: ARRAY<DOUBLE>>",
    )
)

rows = spark.sql(f"SELECT latitude, longitude FROM {CITY_DIM_TABLE}").collect()
locations = [Location(row.latitude, row.longitude) for row in rows]

forecasts = await fetch_hourly_forecast(locations)

spark.sql(f"CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} ({BRONZE_COLUMNS})")

spark.createDataFrame(
    [f.model_dump() for f in forecasts], STAGING_COLUMNS
).createOrReplaceTempView("forecast_staging")

spark.sql(f"""
    INSERT INTO {BRONZE_TABLE}
    SELECT
        f.latitude,
        f.longitude,
        h.forecast_time,
        h.temperature_2m,
        f.hourly_units.temperature_2m,
        h.apparent_temperature,
        f.hourly_units.apparent_temperature,
        h.precipitation,
        f.hourly_units.precipitation,
        h.wind_speed_10m,
        f.hourly_units.wind_speed_10m,
        h.weather_code,
        current_timestamp()
    FROM forecast_staging AS f
    CROSS JOIN LATERAL inline_outer(
        arrays_zip(
            f.hourly.time,
            f.hourly.temperature_2m,
            f.hourly.apparent_temperature,
            f.hourly.precipitation,
            f.hourly.wind_speed_10m,
            f.hourly.weather_code
        )
    ) AS h (
        forecast_time,
        temperature_2m,
        apparent_temperature,
        precipitation,
        wind_speed_10m,
        weather_code
    )
""")